✅ Fase 1: Preparação para Embeddings
Plano:

Carregar todos os .md da SSOT.

Pré-processar texto (limpeza mínima, se necessário).

Gerar embeddings usando um modelo confiável (ex.: sentence-transformers).

Armazenar os embeddings em uma estrutura limpa (novo chroma_ssot_a3).

Indexar para futuras consultas.



In [1]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np

# Caminhos
ssot_dir = r"C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\data\ssot_a3"
emb_dir = r"C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\data\chroma_ssot_a3"
os.makedirs(emb_dir, exist_ok=True)

# Modelo de embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

# Lista para log
log = []

# Processar arquivos
for raiz, _, arquivos in os.walk(ssot_dir):
    for f in arquivos:
        if f.lower().endswith(".md"):
            caminho = os.path.join(raiz, f)
            try:
                with open(caminho, "r", encoding="utf-8") as arq:
                    texto = arq.read()
                
                # Quebra simples em chunks (se necessário no futuro, podemos melhorar)
                chunks = [texto[i:i+1000] for i in range(0, len(texto), 1000)]
                
                # Gerar embeddings
                embeddings = model.encode(chunks, convert_to_numpy=True)

                # Salvar embeddings como .npy
                emb_path = os.path.join(emb_dir, f"{os.path.splitext(f)[0]}.npy")
                np.save(emb_path, embeddings)

                log.append([f, caminho, emb_path, len(chunks), "OK"])

            except Exception as e:
                log.append([f, caminho, "", 0, f"ERRO: {e}"])

# Salvar log
log_csv = os.path.join(r"C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\desenvolvimento", "embeddings_index.csv")
pd.DataFrame(log, columns=["arquivo", "origem", "embedding_armazenado", "chunks", "status"]).to_csv(log_csv, index=False, encoding="utf-8-sig")

print(f"Embeddings gerados e salvos em: {emb_dir}")
print(f"Log salvo em: {log_csv}")


c:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\wilso\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an a

Embeddings gerados e salvos em: C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\data\chroma_ssot_a3
Log salvo em: C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\desenvolvimento\embeddings_index.csv


In [1]:
import os
import pandas as pd
import numpy as np
import chromadb

# Caminhos
emb_dir = r"C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\data\chroma_ssot_a3"
index_dir = r"C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\data\chroma_index"
os.makedirs(index_dir, exist_ok=True)

# Inicializar ChromaDB
client = chromadb.PersistentClient(path=index_dir)
collection = client.get_or_create_collection(name="a3_embeddings")

# Log de indexação
log = []

# Indexar embeddings
for f in os.listdir(emb_dir):
    if f.endswith(".npy"):
        emb_path = os.path.join(emb_dir, f)
        embeddings = np.load(emb_path)

        # Criar IDs únicos para cada chunk
        ids = [f"{f}_{i}" for i in range(len(embeddings))]

        # Adicionar ao Chroma
        collection.add(
            ids=ids,
            embeddings=embeddings.tolist(),
            documents=[f"{f} - chunk {i}" for i in range(len(embeddings))],
            metadatas=[{"arquivo": f, "chunk": i} for i in range(len(embeddings))]
        )

        log.append([f, len(embeddings), emb_path, "INDEXADO"])

# Salvar log
log_csv = os.path.join(r"C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\desenvolvimento", "chroma_index_log.csv")
pd.DataFrame(log, columns=["arquivo", "chunks_indexados", "origem_embedding", "status"]).to_csv(log_csv, index=False, encoding="utf-8-sig")

print(f"Base ChromaDB criada em: {index_dir}")
print(f"Log salvo em: {log_csv}")


Base ChromaDB criada em: C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\data\chroma_index
Log salvo em: C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\desenvolvimento\chroma_index_log.csv
